# Tools

Various utility scripts implemented as notebook cells.

In [2]:
from io import StringIO
import os
import sys

In [3]:
from grep import grep

In [4]:
import home

📁 Current working directory: /home/fuzzy/projects/C++ 2026/hello-world


In [5]:
from tools import *

## grep all files

In [6]:
%%bash
grep -E -n "MINIAUDIO_IMPLEMENTATION" peroxide/*

peroxide/clip.cpp:6:#define MINIAUDIO_IMPLEMENTATION
peroxide/peroxide.hpp:7:#define MINIAUDIO_IMPLEMENTATION
peroxide/player.hpp:7:#define MINIAUDIO_IMPLEMENTATION


## Glob sources for `meson.build`

### Working cell

In [ ]:
from pathlib import Path

# ---------- Configuration ----------
src_dir = Path('source')
meson_file = Path('meson.build')

# Extensions to include
exts = ['.c', '.cp', '.cc', '.cpp']

# ---------- Step 1: Scan src/ ----------
src_files = sorted(f.as_posix() for f in src_dir.iterdir() if f.suffix in exts)

if not src_files:
    print("No source files found in src/.")
else:
    print(f"Found {len(src_files)} source files:")
    for f in src_files:
        print(f"  {f}")

# ---------- Step 2: Update meson.build ----------
# We'll look for a placeholder line: '# --- AUTOGEN_SRC_START ---'
# and replace the block until '# --- AUTOGEN_SRC_END ---'
# Users need to put these markers in meson.build

if not meson_file.exists():
    raise FileNotFoundError(f"{meson_file} not found.")

with meson_file.open('r') as f:
    lines = f.readlines()

new_lines = []
inside_block = False

for line in lines:
    if '# --- AUTOGEN_SRC_START ---' in line:
        inside_block = True
        new_lines.append(line)
        # insert new src_files list here
        new_lines.append('src_files = [\n')
        for fpath in src_files:
            new_lines.append(f"  '{fpath}',\n")
        new_lines.append(']\n')
        continue
    if '# --- AUTOGEN_SRC_END ---' in line:
        inside_block = False
    if not inside_block:
        new_lines.append(line)

with meson_file.open('w') as f:
    f.writelines(new_lines)

print(f"meson.build updated with {len(src_files)} source files.")


### Setup

Assume this directory structure:

```
├── bin
├── data
│   └── data.json
...
├── Doxyfile
├── include
├── LICENSE
├── meson.build
├── README.md
├── source
├── tools
│   ├── glob4meson
│   │   ├── glob4meson.py
│   │   ├── __init__.py
│   │   └── __main__.py
│   └── __init__.py
```

Then, to import from `glob4meson`:

### Execute the Function

In [ ]:
from tools.glob4meson.glob4meson import glob4meson as g4m

In [ ]:
g4m()

```bash
py314 -m tools.glob4meson
```

## Combine Headers

### Algorithm

* 
* Aggregate includes
  * Regex
  * 

In [5]:
[s for s in os.listdir("include") if s.endswith('.hpp') and s != 'sysinc.hpp']

['str.hpp',
 'logging.hpp',
 'program.hpp',
 'config.hpp',
 'types.hpp',
 'fs.hpp',
 'constants.hpp',
 'datetime.hpp',
 'filter.hpp',
 'environment.hpp',
 'globals.hpp']

In [ ]:
FILES = [
    "macros",
    "types",
    "constants",
    "datetime",
    "str",
    "fs",
    "filter",
    "config",
    "environment",
    "logging",
    "globals",
    "program"
]
PREFIX = Path("include")
SUFFIX = "hpp"

def suffix(s:str)->str:
    """ Return `SUFFIX` unless `s` is "macro". """
    return SUFFIX if s != "macros" else "h"

REGEX = "#include"
g = grep(REGEX)
text = EMPTY_STR

for f in FILES:
    p = PREFIX / (f + PERIOD + suffix(f))
    # print(f"{NEWLINE + str(p)}:{NEWLINE}")
    result = (p).read_text() | g
    text += NEWLINE + str(result)

print(text)

CONTENT_MARKER = "// --*-- content marker for hw7.hpp"

lines = sorted([s[9:] for s in text.split(NEWLINE) if s != EMPTY_STR])

lines = [s for s in lines if not Path(s.strip('"')).stem in FILES]

lines = set(lines)

output = """/**
 * @file hw7.hpp
 *
 * Single header for `hw7`.
 */

// # System headers

"""

for s in lines:
    output += f'#include {s + NEWLINE}'
for f in FILES:
    output += Path(PREFIX / (f + PERIOD + suffix(f))).read_text().partition(CONTENT_MARKER)[2]

Path("hw7/hw7.hpp").write_text(output)